# 15 — Manual Agentic GRPO: env → weights → RLCluster → GRPOLearner

Эта тетрадка не прячет обучение за `scripts/run_agentic_grpo.py`. Она вручную собирает все детали, чтобы можно было инспектировать каждый слой:

1. project/profile/config;
2. CrafText runtime и agentic environment;
3. Tunix topology/workload/preflight;
4. Qwen tokenizer/weights assets;
5. `RLCluster(actor, rollout, reference)`;
6. `ToolAgent`, `CrafTextAgenticEnvironment`, `GRPOLearner`;
7. `learner.train(...)`.

Локально всегда можно выполнить dry/preflight и scripted smoke. Ячейки с весами и learner gated через `RUN_REAL_GRPO=1`, чтобы notebook не скачивал и не аллоцировал модель случайно.


## 0. External preparation

Перед real run подготовьте зависимости и локальные веса вне notebook:

```bash
pyenv exec python -m uv sync --extra envs --extra prompts --extra tunix --extra dev
# Qwen snapshot должен существовать локально:
# artifacts/models/qwen25-05b-instruct
```

Notebook не скачивает веса неявно.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import jax

from tunix_craftext.agentic_craftext import (
    CrafTextAgenticEnvironment,
    CrafTextStepTool,
    agentic_environment_kwargs,
)
from tunix_craftext.agentic_grpo_smoke import (
    collect_scripted_grpo_group_sync,
    save_scripted_grpo_smoke,
)
from tunix_craftext.config import load_mvp_config
from tunix_craftext.grpo_profile import build_grpo_evidence_manifest, load_agentic_grpo_profile
from tunix_craftext.preflight import pinned_qwen_tensor_shape, validate_agentic_grpo_preflight
from tunix_craftext.rlcluster_workload import (
    AgenticGrpoWorkloadSpec,
    build_agentic_grpo_cluster,
    load_agentic_grpo_qwen_assets,
)
from tunix_craftext.runtime import build_craftext_runtime
from tunix_craftext.tunix_adapter import load_qwen_hf_tokenizer
from tunix_craftext.tunix_topology import load_tunix_topology, role_to_meshes


In [ ]:
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file()), None)
if ROOT is None:
    raise RuntimeError('Run this notebook from inside the tunix-craftext repository')

PROFILE_PATH = ROOT / 'configs/grpo/qwen_agentic_local.yaml'
SCRIPTED_EVIDENCE = ROOT / 'artifacts/runs/agentic-grpo-scripted-smoke-notebook.json'

RUN_REAL_GRPO = os.environ.get('RUN_REAL_GRPO') == '1'
ALLOW_CPU_SMOKE = os.environ.get('ALLOW_CPU_SMOKE') == '1'

print('repo:', ROOT)
print('jax backend:', jax.default_backend())
print('RUN_REAL_GRPO:', RUN_REAL_GRPO)
print('ALLOW_CPU_SMOKE:', ALLOW_CPU_SMOKE)


## 1. Load profile and MVP config manually

Profile describes the model/topology/workload/evidence paths. MVP config builds the real CrafText runtime.


In [ ]:
profile = load_agentic_grpo_profile(PROFILE_PATH)
config_path = ROOT / profile.environment_config
topology_path = ROOT / profile.topology_config
snapshot = ROOT / profile.model.snapshot

config = load_mvp_config(config_path)
runtime = build_craftext_runtime(config)

print('profile:', profile.run.name)
print('goal:', profile.run.goal)
print('environment config:', config_path, config_path.is_file())
print('topology config:', topology_path, topology_path.is_file())
print('snapshot:', snapshot, snapshot.is_dir())
print('action count:', runtime.action_count)
print('first actions:', runtime.actions.labels[:8])


## 2. Build and inspect one real CrafText agentic environment

This manually constructs the same environment class that Tunix workers use. It proves config → runtime → prompt/tool environment before touching weights.


In [ ]:
env_kwargs = agentic_environment_kwargs(config_path)
task = {
    'goal': profile.run.goal,
    'seed': config.run.seed,
    'horizon': config.environment.horizon,
}
manual_env = CrafTextAgenticEnvironment(task, **env_kwargs, group_id=0, pair_index=0)
observation, info = manual_env.reset()

print('env kwargs:', env_kwargs)
print('reset info:', info)
print('initial prompt preview:')
print(observation['question'][:1000])


## 3. Topology, workload spec and preflight

This is the no-weight allocation gate. It validates role meshes and static batch/cache constraints.


In [ ]:
topology = load_tunix_topology(topology_path)
meshes = role_to_meshes(topology)
spec = profile.workload
validate_agentic_grpo_preflight(topology, spec, pinned_qwen_tensor_shape())

print('topology:', topology.name, topology.role_to_device_indices)
for role, mesh in meshes.items():
    print(role, mesh.shape, mesh.axis_names)
print('workload:', spec)


## 4. Evidence manifest before model allocation

This is useful even when the real trainer is skipped. It records profile SHA, model identity, package versions and paths.


In [ ]:
manifest = build_grpo_evidence_manifest(profile, profile_path=PROFILE_PATH, repo_root=ROOT)
profile.evidence.provenance.parent.mkdir(parents=True, exist_ok=True)
profile.evidence.provenance.write_text(json.dumps(manifest, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print('wrote:', profile.evidence.provenance)
print(json.dumps(manifest['workload'], indent=2, sort_keys=True))


## 5. Local scripted GRPO smoke: same env/tool loop, no model

This runs several generations for one task group through Tunix `TrajectoryCollectEngine`, but uses scripted tool calls instead of LLM generation. It is the local proof that agentic rollout and group rewards work before real model allocation.


In [ ]:
action_sequences = tuple(
    tuple('NOOP' if generation % 2 == 0 else 'LEFT' for _ in range(config.environment.horizon))
    for generation in range(profile.workload.num_generations)
)
scripted_results = collect_scripted_grpo_group_sync(
    config_path=config_path,
    goal=profile.run.goal,
    seed=config.run.seed,
    group_id=0,
    action_sequences=action_sequences,
    horizon=config.environment.horizon,
)
save_scripted_grpo_smoke(SCRIPTED_EVIDENCE, scripted_results)
for result in scripted_results:
    print(result)
print('saved:', SCRIPTED_EVIDENCE)


## 6. Load tokenizer and Qwen actor/reference weights manually

This is the first heavy cell. It is gated because it allocates model memory. It loads actor/reference assets on declared role meshes and keeps tokenizer explicit.


In [ ]:
if not RUN_REAL_GRPO:
    assets = None
    hf_tokenizer = None
    print('Skipping weight loading. Set RUN_REAL_GRPO=1 to load Qwen assets.')
elif not snapshot.is_dir():
    raise FileNotFoundError(f'Local Qwen snapshot is required: {snapshot}')
elif jax.default_backend() == 'cpu' and not ALLOW_CPU_SMOKE:
    raise RuntimeError(
        'Real Agentic Qwen GRPO is intended for accelerator runner. '
        'Set ALLOW_CPU_SMOKE=1 only to reproduce/debug local CPU upstream failure.'
    )
else:
    assets = load_agentic_grpo_qwen_assets(snapshot, topology)
    hf_tokenizer = load_qwen_hf_tokenizer(snapshot)
    print('loaded actor:', type(assets.actor))
    print('loaded reference:', type(assets.reference))
    print('loaded tokenizer:', type(assets.tokenizer))
    print('loaded hf tokenizer:', type(hf_tokenizer))


## 7. Build RLCluster manually

No learner yet: this just creates the public Tunix cluster from explicit assets and config.


In [ ]:
if not RUN_REAL_GRPO:
    cluster = None
    print('Skipping RLCluster construction.')
else:
    assert assets is not None
    cluster = build_agentic_grpo_cluster(topology, spec, assets)
    print('cluster:', type(cluster))
    print('roles:', {role.value: mesh.shape for role, mesh in cluster.cluster_config.role_to_mesh.items()})


## 8. Build ToolAgent + GRPOLearner manually

This is the actual trainer object. It wires the cluster, parser, agent class, tool map and environment class.


In [ ]:
if not RUN_REAL_GRPO:
    learner = None
    print('Skipping GRPOLearner construction.')
else:
    from tunix.rl.agentic.agentic_grpo_learner import GRPOConfig, GRPOLearner
    from tunix.rl.agentic.agents.tool_agent import ToolAgent
    from tunix.rl.agentic.parser.chat_template_parser.parser import QwenChatTemplateParser

    assert cluster is not None and hf_tokenizer is not None
    learner = GRPOLearner(
        rl_cluster=cluster,
        algo_config=GRPOConfig(
            num_generations=profile.workload.num_generations,
            max_response_length=profile.workload.max_new_tokens,
            max_concurrency=profile.workload.max_concurrency,
            system_prompt='Use craftext_step for every environment action.',
        ),
        chat_parser=QwenChatTemplateParser(hf_tokenizer, enable_thinking=False),
        agent_class=ToolAgent,
        agent_kwargs={
            'system_prompt': 'Use craftext_step for every environment action.',
            'tool_parser_name': 'qwen',
            'tool_map': {'craftext_step': CrafTextStepTool},
        },
        env_class=CrafTextAgenticEnvironment,
        env_kwargs=env_kwargs,
    )
    print('learner:', type(learner))


## 9. Train manually

This calls `learner.train(...)` directly. For the current local profile it runs one update/batch; on CPU it is blocked unless `ALLOW_CPU_SMOKE=1`.


In [ ]:
def task_batches(*, goal: str, seed: int, batch_size: int, count: int, horizon: int):
    import numpy as np
    for batch_index in range(count):
        start = seed + batch_index * batch_size
        yield {
            'goal': [goal] * batch_size,
            'seed': np.arange(start, start + batch_size, dtype=np.int32),
            'horizon': np.full(batch_size, horizon, dtype=np.int32),
        }

if not RUN_REAL_GRPO:
    print('Skipping learner.train. Set RUN_REAL_GRPO=1 to execute training.')
else:
    assert learner is not None and cluster is not None
    try:
        learner.train(
            task_batches(
                goal=profile.run.goal,
                seed=config.run.seed,
                batch_size=profile.workload.mini_batch_size,
                count=profile.workload.max_steps,
                horizon=config.environment.horizon,
            ),
            skip_jit=False,
        )
    finally:
        cluster.close()


## 10. Next: PPO-Lag / CPO

Once this GRPO path runs on target hardware, PPO-Lag/CPO should reuse the same agentic environment/tool transport and add:

- value critic for reward return;
- cost critic for safety/cost return;
- Lagrange multiplier update for PPO-Lag;
- constrained objective / projection terms for CPO;
- checkpoint metadata for actor, reference, reward critic, cost critic, optimizer and policy version.
